In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip -qq install --pre torch torchvision torchaudio --extra-index-url "https://download.pytorch.org/whl/nightly/cuda"

In [10]:
!pip -qq install -r "/content/drive/MyDrive/Image and Video Understanding Project/MaskRCNN/requirements.txt"

In [11]:
!pip -qq install 'git+https://github.com/facebookresearch/detectron2.git'

  Preparing metadata (setup.py) ... done


In [22]:
# register dataset
from detectron2.data.datasets import register_coco_instances
from detectron2.data import DatasetCatalog, MetadataCatalog
data_dir = '/content/drive/MyDrive/Image and Video Understanding Project/DATA/COCO'

for name in ["taco_train", "taco_val", "taco_test"]:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

# === REGISTER DATASETS ===
register_coco_instances("taco_train", {}, f"{data_dir}/annotations_train.json", f"{data_dir}/images")
register_coco_instances("taco_test", {}, f"{data_dir}/annotations_test.json", f"{data_dir}/images")
register_coco_instances("taco_val", {}, f"{data_dir}/annotations_val.json", f"{data_dir}/images")



In [18]:
trained_model = "COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml"
class_number = 20
batch_size = 16
iter_number = 1000
images_number_per_batch = 8
prediction_confidence = 0.5
image_size = 640

In [20]:
import copy
import pandas as pd
import os
import torch
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator
import detectron2.data

# Define search space
learning_rates = [0.0001, 0.001, 0.01]
gammas = [0.1, 0.5]
nms_thresholds = [0.6, 0.8]

# Base config
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(trained_model))  # Define `trained_model` before
cfg.DATASETS.TRAIN = ("taco_train",)
cfg.DATASETS.TEST = ("taco_test",)
cfg.DATALOADER.NUM_WORKERS = 4

output_dir = "/content/drive/MyDrive/Image and Video Understanding Project/MaskRCNN/results/tuning"

cfg.SOLVER.IMS_PER_BATCH = images_number_per_batch

# Store results
results = []

# Grid search loop
for lr in learning_rates:
    for gamma in gammas:
        for nms_thresh in nms_thresholds:
            print(f"Training with lr={lr}, gamma={gamma}, nms_thresh={nms_thresh}")

            # === Create new config for each run ===
            cfg_tune = copy.deepcopy(cfg)
            cfg_tune.SOLVER.BASE_LR = lr
            cfg_tune.SOLVER.GAMMA = gamma
            cfg_tune.MODEL.RPN.NMS_THRESH = nms_thresh
            cfg_tune.SOLVER.MAX_ITER = iter_number
            cfg_tune.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = batch_size

            # Image size settings
            cfg_tune.INPUT.MIN_SIZE_TRAIN = (image_size,)
            cfg_tune.INPUT.MAX_SIZE_TRAIN = image_size
            cfg_tune.INPUT.MIN_SIZE_TEST = image_size
            cfg_tune.INPUT.MAX_SIZE_TEST = image_size

            # Output dir
            run_name = f"lr{lr}_gamma{gamma}_nms{nms_thresh}_bsz{batch_size}_img{image_size}"
            cfg_tune.OUTPUT_DIR = os.path.join(output_dir, run_name)
            os.makedirs(cfg_tune.OUTPUT_DIR, exist_ok=True)

            # Train
            trainer = DefaultTrainer(cfg_tune)
            trainer.resume_or_load(resume=False)
            trainer.train()

            # Evaluate
            evaluator = COCOEvaluator("taco_val", output_dir=cfg_tune.OUTPUT_DIR)
            val_loader = detectron2.data.build_detection_test_loader(cfg_tune, "taco_val")
            metrics = detectron2.engine.defaults.inference_on_dataset(trainer.model, val_loader, evaluator)

            # Save results
            results.append({
                "lr": lr,
                "gamma": gamma,
                "nms_thresh": nms_thresh,
                "batch_size": batch_size,
                "image_size": image_size,
                "mAP": metrics["segm"]["AP"]
            })

# Save all results to CSV
pd.DataFrame(results).to_csv("/content/drive/MyDrive/Image and Video Understanding Project/MaskRCNN/results/tuning/tuning_results.csv", index=False)


Training with lr=0.0001, gamma=0.1, nms_thresh=0.6
[08/04 14:21:06 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_f

backbone.fpn_lateral2.{bias, weight}
backbone.fpn_lateral3.{bias, weight}
backbone.fpn_lateral4.{bias, weight}
backbone.fpn_lateral5.{bias, weight}
backbone.fpn_output2.{bias, weight}
backbone.fpn_output3.{bias, weight}
backbone.fpn_output4.{bias, weight}
backbone.fpn_output5.{bias, weight}
proposal_generator.rpn_head.anchor_deltas.{bias, weight}
proposal_generator.rpn_head.conv.{bias, weight}
proposal_generator.rpn_head.objectness_logits.{bias, weight}
roi_heads.box_head.fc1.{bias, weight}
roi_heads.box_head.fc2.{bias, weight}
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
roi_heads.mask_head.deconv.{bias, weight}
roi_heads.mask_head.mask_fcn1.{bias, weight}
roi_heads.mask_head.mask_fcn2.{bias, weight}
roi_heads.mask_head.mask_fcn3.{bias, weight}
roi_heads.mask_head.mask_fcn4.{bias, weight}
roi_heads.mask_head.predictor.{bias, weight}
  fc1000.{bias, weight}


[08/04 14:21:06 d2.engine.train_loop]: Starting training from iteration 0
[08/04 14:21:23 d2.utils.events]:  eta: 0:12:59  iter: 19  total_loss: 6.781  loss_cls: 4.826  loss_box_reg: 0.2712  loss_mask: 0.693  loss_rpn_cls: 0.6847  loss_rpn_loc: 0.188    time: 0.7841  last_time: 0.9807  data_time: 0.4030  last_data_time: 0.5522   lr: 1.9981e-06  max_mem: 5011M
[08/04 14:21:38 d2.utils.events]:  eta: 0:12:52  iter: 39  total_loss: 6.191  loss_cls: 4.247  loss_box_reg: 0.2116  loss_mask: 0.6931  loss_rpn_cls: 0.6824  loss_rpn_loc: 0.2974    time: 0.7745  last_time: 0.5346  data_time: 0.3378  last_data_time: 0.1140   lr: 3.9961e-06  max_mem: 5748M
[08/04 14:21:53 d2.utils.events]:  eta: 0:12:36  iter: 59  total_loss: 4.876  loss_cls: 3.086  loss_box_reg: 0.212  loss_mask: 0.6933  loss_rpn_cls: 0.6749  loss_rpn_loc: 0.2282    time: 0.7694  last_time: 0.8701  data_time: 0.3304  last_data_time: 0.4449   lr: 5.9941e-06  max_mem: 5748M
[08/04 14:22:10 d2.utils.events]:  eta: 0:11:38  iter: 79  

backbone.fpn_lateral2.{bias, weight}
backbone.fpn_lateral3.{bias, weight}
backbone.fpn_lateral4.{bias, weight}
backbone.fpn_lateral5.{bias, weight}
backbone.fpn_output2.{bias, weight}
backbone.fpn_output3.{bias, weight}
backbone.fpn_output4.{bias, weight}
backbone.fpn_output5.{bias, weight}
proposal_generator.rpn_head.anchor_deltas.{bias, weight}
proposal_generator.rpn_head.conv.{bias, weight}
proposal_generator.rpn_head.objectness_logits.{bias, weight}
roi_heads.box_head.fc1.{bias, weight}
roi_heads.box_head.fc2.{bias, weight}
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
roi_heads.mask_head.deconv.{bias, weight}
roi_heads.mask_head.mask_fcn1.{bias, weight}
roi_heads.mask_head.mask_fcn2.{bias, weight}
roi_heads.mask_head.mask_fcn3.{bias, weight}
roi_heads.mask_head.mask_fcn4.{bias, weight}
roi_heads.mask_head.predictor.{bias, weight}
  fc1000.{bias, weight}


[08/04 14:45:45 d2.engine.train_loop]: Starting training from iteration 0
[08/04 14:46:02 d2.utils.events]:  eta: 0:13:17  iter: 19  total_loss: 6.354  loss_cls: 4.377  loss_box_reg: 0.2761  loss_mask: 0.6934  loss_rpn_cls: 0.7012  loss_rpn_loc: 0.2052    time: 0.7898  last_time: 0.9678  data_time: 0.4121  last_data_time: 0.5394   lr: 1.9981e-06  max_mem: 8800M
[08/04 14:46:18 d2.utils.events]:  eta: 0:12:33  iter: 39  total_loss: 5.723  loss_cls: 3.697  loss_box_reg: 0.3387  loss_mask: 0.6929  loss_rpn_cls: 0.6974  loss_rpn_loc: 0.26    time: 0.7843  last_time: 0.7994  data_time: 0.3494  last_data_time: 0.3799   lr: 3.9961e-06  max_mem: 8800M
[08/04 14:46:33 d2.utils.events]:  eta: 0:12:17  iter: 59  total_loss: 4.058  loss_cls: 2.235  loss_box_reg: 0.2642  loss_mask: 0.6932  loss_rpn_cls: 0.6903  loss_rpn_loc: 0.2016    time: 0.7793  last_time: 0.8951  data_time: 0.3375  last_data_time: 0.4665   lr: 5.9941e-06  max_mem: 8800M
[08/04 14:46:50 d2.utils.events]:  eta: 0:11:20  iter: 79 

...

[08/04 18:10:55 d2.engine.train_loop]: Starting training from iteration 0
[08/04 18:11:12 d2.utils.events]:  eta: 0:13:30  iter: 19  total_loss: 4.696  loss_cls: 2.646  loss_box_reg: 0.3434  loss_mask: 0.6928  loss_rpn_cls: 0.6923  loss_rpn_loc: 0.1984    time: 0.8012  last_time: 0.9960  data_time: 0.4086  last_data_time: 0.5551   lr: 0.00019981  max_mem: 9676M
[08/04 18:11:27 d2.utils.events]:  eta: 0:13:04  iter: 39  total_loss: 3.056  loss_cls: 1.367  loss_box_reg: 0.2921  loss_mask: 0.6898  loss_rpn_cls: 0.514  loss_rpn_loc: 0.2097    time: 0.7928  last_time: 0.4423  data_time: 0.3446  last_data_time: 0.0060   lr: 0.00039961  max_mem: 9676M
[08/04 18:11:43 d2.utils.events]:  eta: 0:12:48  iter: 59  total_loss: 2.508  loss_cls: 0.9442  loss_box_reg: 0.2692  loss_mask: 0.6856  loss_rpn_cls: 0.3525  loss_rpn_loc: 0.1425    time: 0.7891  last_time: 0.8940  data_time: 0.3404  last_data_time: 0.4601   lr: 0.00059941  max_mem: 9676M
[08/04 18:12:00 d2.utils.events]:  eta: 0:11:58  iter: 7

backbone.fpn_lateral2.{bias, weight}
backbone.fpn_lateral3.{bias, weight}
backbone.fpn_lateral4.{bias, weight}
backbone.fpn_lateral5.{bias, weight}
backbone.fpn_output2.{bias, weight}
backbone.fpn_output3.{bias, weight}
backbone.fpn_output4.{bias, weight}
backbone.fpn_output5.{bias, weight}
proposal_generator.rpn_head.anchor_deltas.{bias, weight}
proposal_generator.rpn_head.conv.{bias, weight}
proposal_generator.rpn_head.objectness_logits.{bias, weight}
roi_heads.box_head.fc1.{bias, weight}
roi_heads.box_head.fc2.{bias, weight}
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
roi_heads.mask_head.deconv.{bias, weight}
roi_heads.mask_head.mask_fcn1.{bias, weight}
roi_heads.mask_head.mask_fcn2.{bias, weight}
roi_heads.mask_head.mask_fcn3.{bias, weight}
roi_heads.mask_head.mask_fcn4.{bias, weight}
roi_heads.mask_head.predictor.{bias, weight}
  fc1000.{bias, weight}


[08/04 18:47:03 d2.engine.train_loop]: Starting training from iteration 0
[08/04 18:47:20 d2.utils.events]:  eta: 0:13:31  iter: 19  total_loss: 4.153  loss_cls: 2.148  loss_box_reg: 0.1632  loss_mask: 0.6926  loss_rpn_cls: 0.7066  loss_rpn_loc: 0.2034    time: 0.8073  last_time: 0.9977  data_time: 0.4169  last_data_time: 0.5670   lr: 0.00019981  max_mem: 9676M
[08/04 18:47:35 d2.utils.events]:  eta: 0:12:25  iter: 39  total_loss: 2.838  loss_cls: 1.244  loss_box_reg: 0.2349  loss_mask: 0.689  loss_rpn_cls: 0.5168  loss_rpn_loc: 0.2294    time: 0.7854  last_time: 0.4450  data_time: 0.3295  last_data_time: 0.0172   lr: 0.00039961  max_mem: 9676M
[08/04 18:47:51 d2.utils.events]:  eta: 0:12:52  iter: 59  total_loss: 2.525  loss_cls: 0.983  loss_box_reg: 0.311  loss_mask: 0.6846  loss_rpn_cls: 0.3455  loss_rpn_loc: 0.1393    time: 0.7860  last_time: 0.8997  data_time: 0.3479  last_data_time: 0.4687   lr: 0.00059941  max_mem: 9676M
[08/04 18:48:08 d2.utils.events]:  eta: 0:11:47  iter: 79 

backbone.fpn_lateral2.{bias, weight}
backbone.fpn_lateral3.{bias, weight}
backbone.fpn_lateral4.{bias, weight}
backbone.fpn_lateral5.{bias, weight}
backbone.fpn_output2.{bias, weight}
backbone.fpn_output3.{bias, weight}
backbone.fpn_output4.{bias, weight}
backbone.fpn_output5.{bias, weight}
proposal_generator.rpn_head.anchor_deltas.{bias, weight}
proposal_generator.rpn_head.conv.{bias, weight}
proposal_generator.rpn_head.objectness_logits.{bias, weight}
roi_heads.box_head.fc1.{bias, weight}
roi_heads.box_head.fc2.{bias, weight}
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
roi_heads.mask_head.deconv.{bias, weight}
roi_heads.mask_head.mask_fcn1.{bias, weight}
roi_heads.mask_head.mask_fcn2.{bias, weight}
roi_heads.mask_head.mask_fcn3.{bias, weight}
roi_heads.mask_head.mask_fcn4.{bias, weight}
roi_heads.mask_head.predictor.{bias, weight}
  fc1000.{bias, weight}


[08/04 19:19:27 d2.engine.train_loop]: Starting training from iteration 0
[08/04 19:19:44 d2.utils.events]:  eta: 0:14:56  iter: 19  total_loss: 4.348  loss_cls: 2.441  loss_box_reg: 0.2183  loss_mask: 0.6926  loss_rpn_cls: 0.684  loss_rpn_loc: 0.2034    time: 0.8332  last_time: 0.9827  data_time: 0.4143  last_data_time: 0.5503   lr: 0.00019981  max_mem: 9676M
[08/04 19:20:00 d2.utils.events]:  eta: 0:13:01  iter: 39  total_loss: 3.005  loss_cls: 1.259  loss_box_reg: 0.2757  loss_mask: 0.6887  loss_rpn_cls: 0.5142  loss_rpn_loc: 0.2466    time: 0.8001  last_time: 0.4506  data_time: 0.3307  last_data_time: 0.0220   lr: 0.00039961  max_mem: 9676M
[08/04 19:20:15 d2.utils.events]:  eta: 0:12:45  iter: 59  total_loss: 2.45  loss_cls: 0.9569  loss_box_reg: 0.288  loss_mask: 0.6842  loss_rpn_cls: 0.3448  loss_rpn_loc: 0.1448    time: 0.7923  last_time: 0.9069  data_time: 0.3391  last_data_time: 0.4704   lr: 0.00059941  max_mem: 9676M
[08/04 19:20:32 d2.utils.events]:  eta: 0:11:48  iter: 79 

In [25]:
from detectron2.data.datasets import register_coco_instances
from detectron2.data import DatasetCatalog, MetadataCatalog

import os
import torch
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer, hooks
from detectron2.evaluation import COCOEvaluator
import multiprocessing
multiprocessing.set_start_method("spawn", force=True)
output_dir = "/content/drive/MyDrive/Image and Video Understanding Project/MaskRCNN/results/training"


# === CONFIG PARAMETERS ===
trained_model = "COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml"
class_number = 20
batch_size = 16
iter_number = 6000
learning_rate = 0.0001
data_loader_worker_number = 4
images_number_per_batch = 8
prediction_confidence = 0.5
image_size = 960

# === CONFIG SETUP ===
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(trained_model))

cfg.DATASETS.TRAIN = ("taco_train",)
cfg.DATASETS.TEST = ("taco_test",)
cfg.DATALOADER.NUM_WORKERS = data_loader_worker_number

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(trained_model)
cfg.SOLVER.IMS_PER_BATCH = images_number_per_batch
cfg.SOLVER.BASE_LR = lr
cfg.SOLVER.GAMMA = gamma
cfg.MODEL.RPN.NMS_THRESH = nms_thresh
cfg.SOLVER.MAX_ITER = iter_number
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = batch_size
cfg.MODEL.ROI_HEADS.NUM_CLASSES = class_number

cfg.INPUT.MIN_SIZE_TRAIN = (image_size,)
cfg.INPUT.MIN_SIZE_TEST = image_size
cfg.INPUT.MAX_SIZE_TRAIN = image_size
cfg.INPUT.MAX_SIZE_TEST = image_size

# ✅ Set output directory
cfg.OUTPUT_DIR = output_dir
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# === SAVE CONFIG TO FILE ===
with open(os.path.join(cfg.OUTPUT_DIR, "config.yaml"), "w") as f:
    f.write(cfg.dump())

# === DEVICE SETUP ===
cfg.MODEL.DEVICE = "cuda"  # or "cuda" if using GPU

# === TRAIN ===
trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

# === PREDICTION ===
from detectron2.engine import DefaultPredictor

cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = prediction_confidence
cfg.DATASETS.TEST = ("taco_test", )
predictor = DefaultPredictor(cfg)

# === VISUALIZATION ===
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.utils.visualizer import Visualizer
import cv2
import random
import matplotlib.pyplot as plt

# Load the dataset
dataset_dicts = DatasetCatalog.get("taco_test")
metadata = MetadataCatalog.get("taco_test")

for d in random.sample(dataset_dicts, 5):
    im = cv2.imread(d["file_name"])
    outputs = predictor(im)

    print("Predictions:", outputs["instances"].to("cpu"))

    # Visualization
    v = Visualizer(im[:, :, ::-1], metadata=metadata, scale=1.0)
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

    # Show result
    plt.figure(figsize=(10, 10))
    plt.imshow(out.get_image()[:, :, ::-1])
    plt.axis('off')
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [26]:
# === EVALUATE ON TEST SET ===
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

print("\n=== Running COCO Evaluation on taco_test ===")
evaluator = COCOEvaluator("taco_test", cfg, False, output_dir=os.path.join(cfg.OUTPUT_DIR, "eval"))
val_loader = build_detection_test_loader(cfg, "taco_test")
metrics = inference_on_dataset(predictor.model, val_loader, evaluator)

# Optional: Save summary to CSV/JSON
import json
import csv

# Save JSON
with open(os.path.join(cfg.OUTPUT_DIR, "eval", "metrics_summary.json"), "w") as f:
    json.dump(metrics, f, indent=4)



=== Running COCO Evaluation on taco_test ===
WARNING [08/04 23:13:09 d2.evaluation.coco_evaluation]: COCO Evaluator instantiated using config, this is deprecated behavior. Please pass in explicit arguments instead.
WARNING [08/04 23:13:09 d2.data.datasets.coco]: 
Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.

[08/04 23:13:09 d2.data.datasets.coco]: Loaded 136 images in COCO format from /content/drive/MyDrive/Image and Video Understanding Project/DATA/COCO/annotations_test.json
[08/04 23:13:09 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(960, 960), max_size=960, sample_style='choice')]
[08/04 23:13:09 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[08/04 23:13:09 d2.data.common]: Serializing 136 elements to byte tensors and concatenating them all ...
[08/04 23:13:09 d2.data.common]: Serialized dataset takes 0.18 MiB
[08/04 